In [1]:
import os
import json
import math
import copy
import csv
import time

import numpy as np
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset

In [2]:
with open(
    "data_split.json",
    "r"
) as f:
    split_data = json.load(f)


train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]


print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))


# Combine validation and test subjects.
# These subjects were not used to train the generative model.
heldout_subjects = sorted(
    list(val_subjects)
    + list(test_subjects)
)


if len(heldout_subjects) < 200:
    raise RuntimeError(
        "Fewer than 200 held-out subjects are available."
    )


training_overlap = (
    set(train_subjects)
    .intersection(
        set(heldout_subjects)
    )
)


if training_overlap:
    raise RuntimeError(
        "Training and held-out subject overlap detected."
    )


print(
    "Total held-out subjects:",
    len(heldout_subjects)
)

Train: 1000
Validation: 125
Test: 126
Total held-out subjects: 251


In [3]:
DATA_DIR = (
    "Data/"
    "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)


if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"BraTS data directory not found: {DATA_DIR}"
    )


print(
    "BraTS data directory:",
    DATA_DIR
)

BraTS data directory: Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData


In [4]:
def preprocess_t2f(image):

    if image.shape != (240, 240, 155):
        raise ValueError(
            f"Unexpected image shape: {image.shape}"
        )

    # Crop to 208 x 224 x 155
    image = image[16:224, 8:232, :]

    # Pad depth to 160
    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    # True original brain foreground
    brain_mask = image > 0

    if not np.any(brain_mask):
        raise ValueError(
            "No foreground voxels found"
        )

    upper = np.percentile(
        image[brain_mask],
        99.9
    )

    image = np.clip(
        image,
        0,
        upper
    )

    # [0,1]
    image = image / upper

    # [-1,1]
    image = (
        image * 2.0
        - 1.0
    )

    # Explicitly enforce air background
    image[~brain_mask] = -1.0

    return (
        image.astype(np.float32),
        brain_mask.astype(np.bool_)
    )

In [5]:
def preprocess_mask(mask):

    if mask.shape != (240, 240, 155):
        raise ValueError(
            f"Unexpected mask shape: {mask.shape}"
        )

    mask = mask[16:224, 8:232, :]

    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(
        np.int64
    )

In [6]:
def calculate_tumour_entropy(
    image,
    mask,
    num_bins=256
):

    tumour_region = (
        mask > 0
    )

    if not np.any(
        tumour_region
    ):
        raise ValueError(
            "No tumour voxels found"
        )

    # image is stored in [-1,1]
    # Convert back to [0,1]
    image_01 = (
        image + 1.0
    ) / 2.0

    image_01 = np.clip(
        image_01,
        0.0,
        1.0
    )

    tumour_values = (
        image_01[
            tumour_region
        ]
    )

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = (
        hist.astype(
            np.float64
        )
    )

    probabilities = (
        probabilities
        / probabilities.sum()
    )

    probabilities = (
        probabilities[
            probabilities > 0
        ]
    )

    entropy = -np.sum(
        probabilities
        * np.log2(
            probabilities
        )
    )

    return np.float32(
        entropy
    )

In [7]:
class BraTSDataset(Dataset):

    def __init__(
        self,
        subjects,
        data_dir
    ):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(
            self.subjects
        )

    def __getitem__(
        self,
        idx
    ):

        subject = (
            self.subjects[idx]
        )

        subject_path = os.path.join(
            self.data_dir,
            subject
        )

        files = os.listdir(
            subject_path
        )

        t2f_files = [
            f for f in files
            if "t2f" in f.lower()
        ]

        seg_files = [
            f for f in files
            if "seg" in f.lower()
        ]

        if len(t2f_files) == 0:
            raise FileNotFoundError(
                f"No T2f file found for {subject}"
            )

        if len(seg_files) == 0:
            raise FileNotFoundError(
                f"No segmentation found for {subject}"
            )

        image = nib.load(
            os.path.join(
                subject_path,
                t2f_files[0]
            )
        ).get_fdata()

        mask = nib.load(
            os.path.join(
                subject_path,
                seg_files[0]
            )
        ).get_fdata()

        image, brain_mask = (
            preprocess_t2f(
                image
            )
        )

        mask = preprocess_mask(
            mask
        )

        entropy = (
            calculate_tumour_entropy(
                image,
                mask
            )
        )

        image = (
            torch.from_numpy(
                image
            )
            .float()
            .unsqueeze(0)
        )

        brain_mask = (
            torch.from_numpy(
                brain_mask
            )
            .bool()
            .unsqueeze(0)
        )

        mask = (
            torch.from_numpy(
                mask
            )
            .long()
            .unsqueeze(0)
        )

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "brain_mask": brain_mask,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
# ============================================================
# Fixed held-out cohort for conditional generation
# ============================================================

COHORT_SIZE = 200
COHORT_SELECTION_SEED = 2026


BASE_OUTPUT_DIR = "evaluation_200"

OUTPUT_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "conditional_ddpm_v3"
)

CONDITION_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "conditions"
)

MASK_DIR = os.path.join(
    CONDITION_DIR,
    "masks"
)

SUBJECTS_JSON = os.path.join(
    CONDITION_DIR,
    "evaluation_subjects_200.json"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    MASK_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Reuse the same cohort if it was already created
# ------------------------------------------------------------

if os.path.isfile(SUBJECTS_JSON):

    with open(
        SUBJECTS_JSON,
        "r"
    ) as f:
        cohort_data = json.load(f)

    if isinstance(
        cohort_data,
        dict
    ):
        evaluation_subjects = (
            cohort_data["subjects"]
        )
    else:
        evaluation_subjects = cohort_data

    print(
        "Loaded existing evaluation cohort:",
        SUBJECTS_JSON
    )


# ------------------------------------------------------------
# Otherwise select a reproducible random cohort
# ------------------------------------------------------------

else:

    rng = np.random.default_rng(
        COHORT_SELECTION_SEED
    )

    selected_indices = rng.choice(
        len(heldout_subjects),
        size=COHORT_SIZE,
        replace=False
    )

    evaluation_subjects = [
        heldout_subjects[int(index)]
        for index in selected_indices
    ]

    cohort_data = {
        "cohort_size":
            COHORT_SIZE,

        "selection_seed":
            COHORT_SELECTION_SEED,

        "source_sets": [
            "validation",
            "test"
        ],

        "subjects":
            evaluation_subjects
    }

    with open(
        SUBJECTS_JSON,
        "w"
    ) as f:
        json.dump(
            cohort_data,
            f,
            indent=2
        )

    print(
        "Created evaluation cohort:",
        SUBJECTS_JSON
    )


# ------------------------------------------------------------
# Validate the selected cohort
# ------------------------------------------------------------

if len(evaluation_subjects) != COHORT_SIZE:
    raise RuntimeError(
        f"Expected {COHORT_SIZE} subjects, "
        f"found {len(evaluation_subjects)}."
    )


if len(set(evaluation_subjects)) != COHORT_SIZE:
    raise RuntimeError(
        "Duplicate subjects found in evaluation cohort."
    )


if not set(evaluation_subjects).issubset(
    set(heldout_subjects)
):
    raise RuntimeError(
        "Evaluation cohort contains non-held-out subjects."
    )


evaluation_dataset = BraTSDataset(
    subjects=evaluation_subjects,
    data_dir=DATA_DIR
)


print(
    "Evaluation cohort size:",
    len(evaluation_dataset)
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Shared condition directory:",
    CONDITION_DIR
)

Loaded existing evaluation cohort: evaluation_200/conditions/evaluation_subjects_200.json
Evaluation cohort size: 200
Synthetic output directory: evaluation_200/conditional_ddpm_v3
Shared condition directory: evaluation_200/conditions


In [9]:
sample = evaluation_dataset[0]


print(
    "Evaluation subject:",
    sample["subject"]
)

print(
    "Image shape:",
    sample["image"].shape
)

print(
    "Brain mask shape:",
    sample["brain_mask"].shape
)

print(
    "Tumour mask shape:",
    sample["mask"].shape
)

print(
    "Mask labels:",
    torch.unique(
        sample["mask"]
    )
)

print(
    "Raw tumour entropy:",
    sample["heterogeneity"].item()
)


assert sample["image"].shape == (
    1,
    208,
    224,
    160
)

assert sample["mask"].shape == (
    1,
    208,
    224,
    160
)

assert torch.any(
    sample["mask"] > 0
)


print(
    "Evaluation condition check passed."
)

Evaluation subject: BraTS-GLI-00085-000
Image shape: torch.Size([1, 208, 224, 160])
Brain mask shape: torch.Size([1, 208, 224, 160])
Tumour mask shape: torch.Size([1, 208, 224, 160])


Mask labels: tensor([0, 1, 2, 3])
Raw tumour entropy: 7.160233020782471
Evaluation condition check passed.


In [10]:
timesteps = 1000


def cosine_beta_schedule(
    timesteps,
    s=0.008
):

    steps = (
        timesteps + 1
    )

    x = torch.linspace(
        0,
        timesteps,
        steps,
        dtype=torch.float64
    )

    alpha_bar = torch.cos(
        (
            (
                x / timesteps
                + s
            )
            / (
                1.0 + s
            )
        )
        * math.pi
        * 0.5
    ) ** 2

    alpha_bar = (
        alpha_bar
        / alpha_bar[0]
    )

    betas = (
        1.0
        - (
            alpha_bar[1:]
            / alpha_bar[:-1]
        )
    )

    return torch.clamp(
        betas,
        min=1e-8,
        max=0.999
    ).float()


def rescale_zero_terminal_snr(
    betas
):

    alphas = (
        1.0 - betas
    )

    alpha_bar = torch.cumprod(
        alphas,
        dim=0
    )

    alpha_bar_sqrt = (
        torch.sqrt(
            alpha_bar
        )
    )

    first = (
        alpha_bar_sqrt[0]
        .clone()
    )

    last = (
        alpha_bar_sqrt[-1]
        .clone()
    )

    alpha_bar_sqrt = (
        alpha_bar_sqrt
        - last
    )

    alpha_bar_sqrt = (
        alpha_bar_sqrt
        * first
        / (
            first - last
        )
    )

    alpha_bar = (
        alpha_bar_sqrt ** 2
    )

    new_alphas = (
        alpha_bar[1:]
        / alpha_bar[:-1]
    )

    new_alphas = torch.cat(
        [
            alpha_bar[0:1],
            new_alphas
        ]
    )

    return (
        1.0
        - new_alphas
    ).float()


betas = cosine_beta_schedule(
    timesteps
)

betas = (
    rescale_zero_terminal_snr(
        betas
    )
)

alphas = (
    1.0 - betas
)

alphas_cumprod = torch.cumprod(
    alphas,
    dim=0
)

alphas_cumprod_prev = F.pad(
    alphas_cumprod[:-1],
    (1, 0),
    value=1.0
)

sqrt_alphas_cumprod = (
    torch.sqrt(
        alphas_cumprod
    )
)

sqrt_one_minus_alphas_cumprod = (
    torch.sqrt(
        1.0
        - alphas_cumprod
    )
)

posterior_variance = (
    betas
    * (
        1.0
        - alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_variance = torch.clamp(
    posterior_variance,
    min=1e-20
)

posterior_mean_coef1 = (
    betas
    * torch.sqrt(
        alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_mean_coef2 = (
    (
        1.0
        - alphas_cumprod_prev
    )
    * torch.sqrt(
        alphas
    )
    / (
        1.0
        - alphas_cumprod
    )
)

snr = (
    alphas_cumprod
    / torch.clamp(
        1.0
        - alphas_cumprod,
        min=1e-12
    )
)


print(
    "Beta range:",
    betas.min().item(),
    betas.max().item()
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

print(
    "Final SNR:",
    snr[-1].item()
)

assert (
    alphas_cumprod[-1].item()
    == 0.0
)

print(
    "Zero-terminal-SNR check passed."
)

Beta range: 4.124641418457031e-05 1.0
Final alpha_cumprod: 0.0
Final SNR: 0.0
Zero-terminal-SNR check passed.


In [11]:
class SinusoidalTimeEmbedding(nn.Module):

    def __init__(
        self,
        dim
    ):
        super().__init__()

        self.dim = dim

    def forward(
        self,
        t
    ):

        half_dim = (
            self.dim // 2
        )

        scale = (
            math.log(10000)
            / (
                half_dim - 1
            )
        )

        embeddings = torch.exp(
            torch.arange(
                half_dim,
                device=t.device
            )
            * -scale
        )

        embeddings = (
            t[:, None].float()
            * embeddings[
                None,
                :
            ]
        )

        embeddings = torch.cat(
            [
                embeddings.sin(),
                embeddings.cos()
            ],
            dim=1
        )

        return embeddings

In [12]:
class ResBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        condition_dim,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=in_channels
        )

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.condition_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                out_channels * 2
            )
        )

        nn.init.zeros_(
            self.condition_mlp[-1].weight
        )

        nn.init.zeros_(
            self.condition_mlp[-1].bias
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.conv2.weight
        )

        nn.init.zeros_(
            self.conv2.bias
        )

        if in_channels != out_channels:

            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )

        else:

            self.residual = (
                nn.Identity()
            )


    def forward(
        self,
        x,
        condition
    ):

        residual = (
            self.residual(
                x
            )
        )

        h = self.norm1(
            x
        )

        h = F.silu(
            h
        )

        h = self.conv1(
            h
        )

        cond = (
            self.condition_mlp(
                condition
            )
        )

        scale, shift = cond.chunk(
            2,
            dim=1
        )

        scale = scale[
            :,
            :,
            None,
            None,
            None
        ]

        shift = shift[
            :,
            :,
            None,
            None,
            None
        ]

        h = self.norm2(
            h
        )

        h = (
            h
            * (
                1.0 + scale
            )
            + shift
        )

        h = F.silu(
            h
        )

        h = self.dropout(
            h
        )

        h = self.conv2(
            h
        )

        return (
            residual + h
        )


class MaskInjection3D(nn.Module):

    def __init__(
        self,
        out_channels,
        strength=0.25
    ):
        super().__init__()

        self.strength = strength

        self.projection = nn.Conv3d(
            3,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.projection.weight
        )

        nn.init.zeros_(
            self.projection.bias
        )


    def forward(
        self,
        x,
        mask_onehot
    ):

        if (
            mask_onehot.shape[2:]
            != x.shape[2:]
        ):

            mask_onehot = F.interpolate(
                mask_onehot,
                size=x.shape[2:],
                mode="nearest"
            )

        mask_feature = (
            self.projection(
                mask_onehot
            )
        )

        mask_feature = (
            self.strength
            * torch.tanh(
                mask_feature
            )
        )

        return (
            x + mask_feature
        )


class AttentionBlock3D(nn.Module):

    def __init__(
        self,
        channels,
        num_heads=4
    ):
        super().__init__()

        self.norm = nn.GroupNorm(
            num_groups=8,
            num_channels=channels
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )


    def forward(
        self,
        x
    ):

        b, c, d, h, w = (
            x.shape
        )

        residual = x

        x = self.norm(
            x
        )

        x = (
            x.permute(
                0,
                2,
                3,
                4,
                1
            )
            .reshape(
                b,
                d * h * w,
                c
            )
        )

        x, _ = self.attention(
            x,
            x,
            x,
            need_weights=False
        )

        x = (
            x.reshape(
                b,
                d,
                h,
                w,
                c
            )
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .contiguous()
        )

        return (
            residual + x
        )

In [13]:
class DownBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        condition_dim
    ):
        super().__init__()

        self.res1 = ResBlock3D(
            in_channels,
            out_channels,
            condition_dim
        )

        self.res2 = ResBlock3D(
            out_channels,
            out_channels,
            condition_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )


    def forward(
        self,
        x,
        condition
    ):

        x = self.res1(
            x,
            condition
        )

        x = self.res2(
            x,
            condition
        )

        skip = x

        x = self.downsample(
            x
        )

        return (
            skip,
            x
        )


class UpBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        condition_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.res1 = ResBlock3D(
            out_channels
            + skip_channels,
            out_channels,
            condition_dim
        )

        self.res2 = ResBlock3D(
            out_channels,
            out_channels,
            condition_dim
        )


    def forward(
        self,
        x,
        skip,
        condition
    ):

        x = self.upsample(
            x
        )

        if (
            x.shape[2:]
            != skip.shape[2:]
        ):
            raise ValueError(
                f"Upsample shape "
                f"{x.shape} != "
                f"skip shape "
                f"{skip.shape}"
            )

        x = torch.cat(
            [
                x,
                skip
            ],
            dim=1
        )

        x = self.res1(
            x,
            condition
        )

        x = self.res2(
            x,
            condition
        )

        return x

In [14]:
class ConditionalUNet3D(nn.Module):

    def __init__(
        self,
        image_channels=1,
        out_channels=1,
        base_channels=16,
        condition_dim=256,
        entropy_scale=0.1
    ):
        super().__init__()

        self.entropy_scale = (
            entropy_scale
        )

        # --------------------------------
        # Time embedding
        # --------------------------------

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(
                condition_dim
            ),
            nn.Linear(
                condition_dim,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        # --------------------------------
        # Entropy embedding
        # --------------------------------

        self.entropy_embedding = nn.Sequential(
            nn.Linear(
                1,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].weight
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].bias
        )

        # --------------------------------
        # Image stem
        # --------------------------------

        self.input_conv = nn.Conv3d(
            image_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        # --------------------------------
        # Multi-scale tumour-mask control
        # --------------------------------

        self.mask0 = MaskInjection3D(
            base_channels
        )

        self.mask1 = MaskInjection3D(
            base_channels * 2
        )

        self.mask2 = MaskInjection3D(
            base_channels * 4
        )

        self.mask3 = MaskInjection3D(
            base_channels * 8
        )

        self.mask4 = MaskInjection3D(
            base_channels * 16
        )

        # --------------------------------
        # Encoder
        # --------------------------------

        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            condition_dim
        )

        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            condition_dim
        )

        self.down3 = DownBlock3D(
            base_channels * 4,
            base_channels * 8,
            condition_dim
        )

        self.down4 = DownBlock3D(
            base_channels * 8,
            base_channels * 16,
            condition_dim
        )

        # --------------------------------
        # Bottleneck
        # --------------------------------

        self.mid1 = ResBlock3D(
            base_channels * 16,
            base_channels * 16,
            condition_dim
        )

        self.mid_attention = AttentionBlock3D(
            base_channels * 16,
            num_heads=4
        )

        self.mid2 = ResBlock3D(
            base_channels * 16,
            base_channels * 16,
            condition_dim
        )

        # --------------------------------
        # Decoder
        # --------------------------------

        self.up4 = UpBlock3D(
            in_channels=base_channels * 16,
            skip_channels=base_channels * 16,
            out_channels=base_channels * 8,
            condition_dim=condition_dim
        )

        self.up3 = UpBlock3D(
            in_channels=base_channels * 8,
            skip_channels=base_channels * 8,
            out_channels=base_channels * 4,
            condition_dim=condition_dim
        )

        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            condition_dim=condition_dim
        )

        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            condition_dim=condition_dim
        )

        self.mask_up4 = MaskInjection3D(
            base_channels * 8
        )

        self.mask_up3 = MaskInjection3D(
            base_channels * 4
        )

        self.mask_up2 = MaskInjection3D(
            base_channels * 2
        )

        self.mask_up1 = MaskInjection3D(
            base_channels
        )

        self.output_norm = nn.GroupNorm(
            num_groups=8,
            num_channels=base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.output_conv.weight
        )

        nn.init.zeros_(
            self.output_conv.bias
        )


    def forward(
        self,
        x,
        t,
        mask,
        heterogeneity
    ):

        # --------------------------------
        # Multi-class tumour condition
        # --------------------------------

        mask = mask.squeeze(
            1
        )

        mask_onehot = F.one_hot(
            mask.long(),
            num_classes=4
        )

        mask_onehot = (
            mask_onehot
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .float()
        )

        # Remove background class
        # -> 3 tumour channels
        mask_onehot = (
            mask_onehot[
                :,
                1:,
                ...
            ]
        )

        # --------------------------------
        # Global condition
        # --------------------------------

        t_emb = self.time_embedding(
            t
        )

        heterogeneity = (
            heterogeneity
            .float()
            .view(
                -1,
                1
            )
        )

        h_emb = self.entropy_embedding(
            heterogeneity
        )

        # Entropy is deliberately weaker
        condition = (
            t_emb
            +
            self.entropy_scale
            * h_emb
        )

        # --------------------------------
        # Encoder
        # --------------------------------

        x = self.input_conv(
            x
        )

        x = self.mask0(
            x,
            mask_onehot
        )

        skip1, x = self.down1(
            x,
            condition
        )

        x = self.mask1(
            x,
            mask_onehot
        )

        skip2, x = self.down2(
            x,
            condition
        )

        x = self.mask2(
            x,
            mask_onehot
        )

        skip3, x = self.down3(
            x,
            condition
        )

        x = self.mask3(
            x,
            mask_onehot
        )

        skip4, x = self.down4(
            x,
            condition
        )

        x = self.mask4(
            x,
            mask_onehot
        )

        # --------------------------------
        # Bottleneck
        # --------------------------------

        x = self.mid1(
            x,
            condition
        )

        x = self.mid_attention(
            x
        )

        x = self.mid2(
            x,
            condition
        )

        # --------------------------------
        # Decoder
        # --------------------------------

        x = self.up4(
            x,
            skip4,
            condition
        )

        x = self.mask_up4(
            x,
            mask_onehot
        )

        x = self.up3(
            x,
            skip3,
            condition
        )

        x = self.mask_up3(
            x,
            mask_onehot
        )

        x = self.up2(
            x,
            skip2,
            condition
        )

        x = self.mask_up2(
            x,
            mask_onehot
        )

        x = self.up1(
            x,
            skip1,
            condition
        )

        x = self.mask_up1(
            x,
            mask_onehot
        )

        x = self.output_norm(
            x
        )

        x = F.silu(
            x
        )

        return self.output_conv(
            x
        )

In [15]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if device.type != "cuda":
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Do not run full 3D Conditional DDPM "
        "sampling on CPU."
    )


print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

Device: cuda
GPU: NVIDIA A40


In [16]:
class EMA:

    def __init__(
        self,
        model,
        decay=0.9999
    ):

        self.decay = decay

        self.ema_model = copy.deepcopy(
            model
        )

        self.ema_model.eval()

        for parameter in (
            self.ema_model.parameters()
        ):
            parameter.requires_grad = False


def load_checkpoint(
    model,
    ema,
    path,
    device
):

    checkpoint = torch.load(
        path,
        map_location=device
    )


    required_keys = {
        "epoch",
        "model_state_dict",
        "ema_state_dict",
        "entropy_mean",
        "entropy_std"
    }


    missing_keys = (
        required_keys
        - set(checkpoint.keys())
    )


    if missing_keys:
        raise KeyError(
            "Checkpoint is missing keys: "
            f"{sorted(missing_keys)}"
        )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    ema.ema_model.load_state_dict(
        checkpoint[
            "ema_state_dict"
        ]
    )


    entropy_mean = float(
        checkpoint[
            "entropy_mean"
        ]
    )


    entropy_std = float(
        checkpoint[
            "entropy_std"
        ]
    )


    if entropy_std <= 0:
        raise RuntimeError(
            "Invalid entropy standard deviation: "
            f"{entropy_std}"
        )


    return (
        int(checkpoint["epoch"]),
        entropy_mean,
        entropy_std
    )

In [17]:
@torch.no_grad()
def sample_conditional_ddpm(
    model,
    shape,
    mask,
    heterogeneity,
    device
):

    model.eval()

    mask = mask.to(
        device
    )

    heterogeneity = (
        heterogeneity
        .to(device)
        .float()
    )

    heterogeneity = (
        heterogeneity
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    sqrt_alpha_bar = (
        sqrt_alphas_cumprod
        .to(device)
    )

    sqrt_one_minus_alpha_bar = (
        sqrt_one_minus_alphas_cumprod
        .to(device)
    )

    coef1 = (
        posterior_mean_coef1
        .to(device)
    )

    coef2 = (
        posterior_mean_coef2
        .to(device)
    )

    posterior_var = (
        posterior_variance
        .to(device)
    )


    # Pure Gaussian terminal prior
    x = torch.randn(
        shape,
        device=device
    )


    for t in reversed(
        range(timesteps)
    ):

        t_batch = torch.full(
            (
                shape[0],
            ),
            t,
            device=device,
            dtype=torch.long
        )

        v_pred = model(
            x,
            t_batch,
            mask,
            heterogeneity
        )


        x0_pred = (
            sqrt_alpha_bar[t]
            * x
            -
            sqrt_one_minus_alpha_bar[t]
            * v_pred
        )

        x0_pred = torch.clamp(
            x0_pred,
            -1.0,
            1.0
        )


        model_mean = (
            coef1[t]
            * x0_pred
            +
            coef2[t]
            * x
        )


        if t > 0:

            noise = torch.randn_like(
                x
            )

            x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[t]
                )
                * noise
            )

        else:

            x = model_mean


    return torch.clamp(
        x,
        -1.0,
        1.0
    )

In [18]:
# ============================================================
# Load final Conditional DDPM V3 checkpoint
# ============================================================

CKPT_PATH = (
    "conditional_v3_checkpoints/"
    "conditional_ddpm_v3_epoch_050.pt"
)


if not os.path.isfile(CKPT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_PATH}"
    )


model = ConditionalUNet3D(
    image_channels=1,
    out_channels=1,
    base_channels=16,
    condition_dim=256,
    entropy_scale=0.1
).to(device)


ema = EMA(
    model,
    decay=0.9999
)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


(
    loaded_epoch,
    ENTROPY_MEAN,
    ENTROPY_STD
) = load_checkpoint(
    model=model,
    ema=ema,
    path=CKPT_PATH,
    device=device
)


if loaded_epoch != 50:
    raise RuntimeError(
        "Expected epoch 50 checkpoint, "
        f"but loaded epoch {loaded_epoch}."
    )


sampling_model = ema.ema_model
sampling_model.eval()


# The ordinary model copy is no longer required.
del model


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "Loaded Conditional DDPM V3 epoch:",
    loaded_epoch
)

print(
    "Checkpoint:",
    CKPT_PATH
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Entropy mean:",
    ENTROPY_MEAN
)

print(
    "Entropy std:",
    ENTROPY_STD
)

print(
    "Sampling model: EMA"
)

Loaded Conditional DDPM V3 epoch: 50
Checkpoint: conditional_v3_checkpoints/conditional_ddpm_v3_epoch_050.pt
Total parameters: 28,832,721
Entropy mean: 6.870205879211426
Entropy std: 0.33203670382499695
Sampling model: EMA


In [19]:
# ============================================================
# Conditional DDPM V3 generation configuration
# ============================================================

NUM_TO_GENERATE = 200

BASE_SEED = 20000


SAMPLE_SHAPE = (
    1,
    1,
    208,
    224,
    160
)


METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "metadata_conditional_ddpm_v3.csv"
)


AFFINE = np.eye(
    4,
    dtype=np.float32
)


if NUM_TO_GENERATE < 1:
    raise ValueError(
        "NUM_TO_GENERATE must be at least 1."
    )


if NUM_TO_GENERATE > COHORT_SIZE:
    raise ValueError(
        "NUM_TO_GENERATE cannot exceed "
        f"COHORT_SIZE={COHORT_SIZE}."
    )


print(
    "Number to generate:",
    NUM_TO_GENERATE
)

print(
    "Available condition subjects:",
    len(evaluation_dataset)
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Condition mask directory:",
    MASK_DIR
)

print(
    "Metadata:",
    METADATA_PATH
)

print(
    "Seed range:",
    BASE_SEED,
    "to",
    BASE_SEED + NUM_TO_GENERATE - 1
)

Number to generate: 200
Available condition subjects: 200
Synthetic output directory: evaluation_200/conditional_ddpm_v3
Condition mask directory: evaluation_200/conditions/masks
Metadata: evaluation_200/conditional_ddpm_v3/metadata_conditional_ddpm_v3.csv
Seed range: 20000 to 20199


In [20]:
# ============================================================
# Generate 200 Conditional DDPM V3 volumes
# ============================================================

metadata_exists = os.path.isfile(
    METADATA_PATH
)


existing_metadata_ids = set()


if metadata_exists:

    with open(
        METADATA_PATH,
        "r",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:
            existing_metadata_ids.add(
                row["sample_id"]
            )


else:

    with open(
        METADATA_PATH,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "sample_id",
            "model",
            "source_subject",
            "filename",
            "condition_mask_filename",
            "seed",
            "raw_entropy",
            "z_entropy",
            "shape_x",
            "shape_y",
            "shape_z",
            "min",
            "max",
            "mean",
            "std",
            "generation_seconds",
            "status"
        ])


def append_metadata(
    sample_id,
    subject,
    filename,
    mask_filename,
    seed,
    raw_entropy,
    z_entropy,
    volume,
    generation_seconds,
    status
):

    with open(
        METADATA_PATH,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            sample_id,
            "conditional_ddpm_v3",
            subject,
            filename,
            mask_filename,
            seed,
            raw_entropy,
            z_entropy,
            volume.shape[0],
            volume.shape[1],
            volume.shape[2],
            float(volume.min()),
            float(volume.max()),
            float(volume.mean()),
            float(volume.std()),
            generation_seconds,
            status
        ])


total_start = time.perf_counter()
generated_this_run = 0


for i in range(
    NUM_TO_GENERATE
):

    sample_id = f"{i:04d}"

    seed = BASE_SEED + i


    # Load one held-out condition.
    sample = evaluation_dataset[i]

    subject = sample[
        "subject"
    ]


    raw_entropy = float(
        sample[
            "heterogeneity"
        ].item()
    )


    z_entropy = (
        raw_entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    filename = (
        f"conditional_ddpm_v3_"
        f"{sample_id}.nii.gz"
    )


    output_path = os.path.join(
        OUTPUT_DIR,
        filename
    )


    mask_filename = (
        f"condition_mask_"
        f"{sample_id}.nii.gz"
    )


    mask_path = os.path.join(
        MASK_DIR,
        mask_filename
    )


    # --------------------------------------------------------
    # Save the shared multi-class condition mask
    # --------------------------------------------------------

    if not os.path.isfile(
        mask_path
    ):

        mask_np = (
            sample["mask"][0]
            .detach()
            .cpu()
            .numpy()
            .astype(np.uint8)
        )


        mask_nifti = nib.Nifti1Image(
            mask_np,
            AFFINE
        )


        mask_nifti.set_data_dtype(
            np.uint8
        )


        nib.save(
            mask_nifti,
            mask_path
        )


        del mask_np
        del mask_nifti


    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------

    if os.path.isfile(
        output_path
    ):

        print(
            f"[{i + 1:03d}/{NUM_TO_GENERATE}] "
            f"{filename} already exists -> skipped"
        )


        # Recover metadata if a volume exists but its row
        # was not written before a previous job stopped.
        if sample_id not in existing_metadata_ids:

            existing_volume = np.asarray(
                nib.load(
                    output_path
                ).dataobj,
                dtype=np.float32
            )


            append_metadata(
                sample_id=sample_id,
                subject=subject,
                filename=filename,
                mask_filename=mask_filename,
                seed=seed,
                raw_entropy=raw_entropy,
                z_entropy=z_entropy,
                volume=existing_volume,
                generation_seconds="",
                status="recovered_existing"
            )


            existing_metadata_ids.add(
                sample_id
            )


            del existing_volume


        del sample
        continue


    print()

    print(
        f"[{i + 1:03d}/{NUM_TO_GENERATE}] "
        f"Generating {filename}"
    )

    print(
        "Source subject:",
        subject
    )

    print(
        "Seed:",
        seed
    )

    print(
        "Raw entropy:",
        raw_entropy
    )

    print(
        "Z entropy:",
        z_entropy
    )


    # --------------------------------------------------------
    # Reproducible seed
    # --------------------------------------------------------

    torch.manual_seed(
        seed
    )

    np.random.seed(
        seed
    )


    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


    # Dataset mask shape:
    # [1, 208, 224, 160]
    #
    # Model mask shape:
    # [1, 1, 208, 224, 160]
    mask_batch = (
        sample["mask"]
        .unsqueeze(0)
    )


    # Dataset entropy shape:
    # scalar
    #
    # Model entropy shape:
    # [1]
    entropy_batch = (
        sample["heterogeneity"]
        .unsqueeze(0)
    )


    # --------------------------------------------------------
    # Generate one complete 3D volume
    # --------------------------------------------------------

    sample_start = time.perf_counter()


    generated = sample_conditional_ddpm(
        model=sampling_model,
        shape=SAMPLE_SHAPE,
        mask=mask_batch,
        heterogeneity=entropy_batch,
        device=device
    )


    if device.type == "cuda":
        torch.cuda.synchronize()


    sample_seconds = (
        time.perf_counter()
        - sample_start
    )


    # --------------------------------------------------------
    # Convert model output from [-1,1] to [0,1]
    # --------------------------------------------------------

    volume = (
        generated[
            0,
            0
        ]
        .detach()
        .float()
        .cpu()
        .numpy()
    )


    volume = (
        volume + 1.0
    ) / 2.0


    volume = np.clip(
        volume,
        0.0,
        1.0
    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # Technical checks
    # --------------------------------------------------------

    expected_shape = (
        208,
        224,
        160
    )


    if volume.shape != expected_shape:
        raise RuntimeError(
            "Unexpected generated shape: "
            f"{volume.shape}"
        )


    if not np.all(
        np.isfinite(volume)
    ):
        raise RuntimeError(
            "NaN or Inf found in "
            f"sample {sample_id}"
        )


    # --------------------------------------------------------
    # Save synthetic NIfTI
    # --------------------------------------------------------

    synthetic_nifti = nib.Nifti1Image(
        volume,
        AFFINE
    )


    synthetic_nifti.set_data_dtype(
        np.float32
    )


    nib.save(
        synthetic_nifti,
        output_path
    )


    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    if sample_id not in existing_metadata_ids:

        append_metadata(
            sample_id=sample_id,
            subject=subject,
            filename=filename,
            mask_filename=mask_filename,
            seed=seed,
            raw_entropy=raw_entropy,
            z_entropy=z_entropy,
            volume=volume,
            generation_seconds=sample_seconds,
            status="generated"
        )


        existing_metadata_ids.add(
            sample_id
        )


    generated_this_run += 1


    completed_files = len([
        name
        for name in os.listdir(
            OUTPUT_DIR
        )
        if (
            name.startswith(
                "conditional_ddpm_v3_"
            )
            and name.endswith(
                ".nii.gz"
            )
        )
    ])


    completed_files = min(
        completed_files,
        NUM_TO_GENERATE
    )


    remaining = (
        NUM_TO_GENERATE
        - completed_files
    )


    estimated_remaining_hours = (
        remaining
        * sample_seconds
        / 3600.0
    )


    print(
        "Saved:",
        output_path
    )

    print(
        "Condition mask:",
        mask_path
    )

    print(
        "Shape:",
        volume.shape
    )

    print(
        "Range:",
        float(volume.min()),
        float(volume.max())
    )

    print(
        "Mean:",
        float(volume.mean())
    )

    print(
        "Std:",
        float(volume.std())
    )

    print(
        f"Generation time: "
        f"{sample_seconds / 60:.2f} min"
    )

    print(
        f"Completed: "
        f"{completed_files}/"
        f"{NUM_TO_GENERATE}"
    )

    print(
        f"Estimated remaining time: "
        f"{estimated_remaining_hours:.2f} h"
    )


    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del sample
    del mask_batch
    del entropy_batch
    del generated
    del volume
    del synthetic_nifti


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


total_seconds = (
    time.perf_counter()
    - total_start
)


print()

print(
    "========================================"
)

print(
    "Conditional DDPM V3 generation finished"
)

print(
    "========================================"
)

print(
    "Generated during this run:",
    generated_this_run
)

print(
    f"Runtime this session: "
    f"{total_seconds / 3600:.2f} h"
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Shared condition directory:",
    CONDITION_DIR
)

print(
    "Metadata:",
    METADATA_PATH
)

[001/200] conditional_ddpm_v3_0000.nii.gz already exists -> skipped



[002/200] Generating conditional_ddpm_v3_0001.nii.gz
Source subject: BraTS-GLI-00756-000
Seed: 20001
Raw entropy: 7.1754231452941895
Z entropy: 0.9192274907163013


Saved: evaluation_200/conditional_ddpm_v3/conditional_ddpm_v3_0001.nii.gz
Condition mask: evaluation_200/conditions/masks/condition_mask_0001.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.0966869592666626
Std: 0.1681482195854187
Generation time: 10.71 min
Completed: 2/200
Estimated remaining time: 35.33 h



[003/200] Generating conditional_ddpm_v3_0002.nii.gz
Source subject: BraTS-GLI-01215-000
Seed: 20002
Raw entropy: 6.929394245147705
Z entropy: 0.17825850351615066


Saved: evaluation_200/conditional_ddpm_v3/conditional_ddpm_v3_0002.nii.gz
Condition mask: evaluation_200/conditions/masks/condition_mask_0002.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.10743405669927597
Std: 0.1861182153224945
Generation time: 10.66 min
Completed: 3/200
Estimated remaining time: 35.01 h


...

Saved: evaluation_200/conditional_ddpm_v3/conditional_ddpm_v3_0197.nii.gz
Condition mask: evaluation_200/conditions/masks/condition_mask_0197.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.10715305060148239
Std: 0.1915697306394577
Generation time: 10.67 min
Completed: 198/200
Estimated remaining time: 0.36 h



[199/200] Generating conditional_ddpm_v3_0198.nii.gz
Source subject: BraTS-GLI-01368-000
Seed: 20198
Raw entropy: 6.876424789428711
Z entropy: 0.018729586656067068


Saved: evaluation_200/conditional_ddpm_v3/conditional_ddpm_v3_0198.nii.gz
Condition mask: evaluation_200/conditions/masks/condition_mask_0198.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.10948172956705093
Std: 0.18914802372455597
Generation time: 10.67 min
Completed: 199/200
Estimated remaining time: 0.18 h



[200/200] Generating conditional_ddpm_v3_0199.nii.gz
Source subject: BraTS-GLI-00621-000
Seed: 20199
Raw entropy: 6.574536323547363
Z entropy: -0.8904725057742349


Saved: evaluation_200/conditional_ddpm_v3/conditional_ddpm_v3_0199.nii.gz
Condition mask: evaluation_200/conditions/masks/condition_mask_0199.nii.gz
Shape: (208, 224, 160)
Range: 0.0 1.0
Mean: 0.12458398938179016
Std: 0.2141249179840088
Generation time: 10.67 min
Completed: 200/200
Estimated remaining time: 0.00 h

Conditional DDPM V3 generation finished
Generated during this run: 199
Runtime this session: 35.44 h
Synthetic output directory: evaluation_200/conditional_ddpm_v3
Shared condition directory: evaluation_200/conditions
Metadata: evaluation_200/conditional_ddpm_v3/metadata_conditional_ddpm_v3.csv
